## Load dos dados

Primeiro verificamos a distribuição dos dados da amostra. Depois buscamos equalizar a amostra por classe, para depois passar para o treinamento.

In [1]:
import pandas as pd
# 1) Carregamento (ajuste o caminho e o tipo)
df_original = pd.read_csv("df_ml_final.csv")  # se estiver em CSV
df_original = df_original.dropna()
#df = df_original.reset_index(drop=True, inplace=False)

In [2]:
labels = ['muito negativo', 'negativo', 'neutro', 'positivo']
valores = [-1, -0.5, 0, 0.5, 1]
pd.cut(df_original['escala'], bins=valores, labels=labels).value_counts()

escala
negativo          102344
neutro             69560
muito negativo     43246
positivo            3602
Name: count, dtype: int64

In [3]:
negativas = df_original.loc[(df_original['escala'] > -0.5) & (df_original['escala'] <= 0)]
negativas.count()

product          102344
processada_ml    102344
escala           102344
dtype: int64

In [4]:
amt_neg = negativas.sample(45000, random_state=7)
amt_neg.head()

,product,processada_ml,escala
167298,Credit reporting or other personal consumer re...,inquiries credit report recognize,0.00
33224,Credit reporting or other personal consumer re...,write formally request block deletion account ...,-0.05
20692,Credit reporting or other personal consumer re...,usc 1681i subject subsection except provide su...,0.00
33318,Credit reporting or other personal consumer re...,file complaint equifax multiple serious violat...,-0.10
187063,Credit reporting or other personal consumer re...,saferent solutions provide inaccurate informat...,0.00


In [5]:
neutras = df_original.loc[(df_original['escala'] > 0) & (df_original['escala'] <= 0.5)]
neutras.count()

product          69560
processada_ml    69560
escala           69560
dtype: int64

In [6]:
neutras = neutras.sample(47000, random_state=7)
neutras.head()

,product,processada_ml,escala
148848,Mortgage,home loan rocket mortgage close refinance rock...,0.10
4551,Credit reporting or other personal consumer re...,provision fair credit report act demand items ...,0.10
16252,Credit reporting or other personal consumer re...,discover several unauthorized inquiries credit...,0.10
9360,Credit reporting or other personal consumer re...,file ftc identity theft affidavit report fourt...,0.10
66201,Credit reporting or other personal consumer re...,entries late payments appear report believe in...,0.05


In [7]:
positivas = df_original.loc[(df_original['escala'] > 0.5)]
positivas.count()

product          3602
processada_ml    3602
escala           3602
dtype: int64

In [8]:
m_neg = df_original.loc[(df_original['escala'] < -0.5)]
m_neg.count()

product          43246
processada_ml    43246
escala           43246
dtype: int64

In [9]:
df = pd.concat([amt_neg, m_neg, neutras, positivas, positivas, positivas], ignore_index=True)
df.head()

,product,processada_ml,escala
0,Credit reporting or other personal consumer re...,inquiries credit report recognize,0.00
1,Credit reporting or other personal consumer re...,write formally request block deletion account ...,-0.05
2,Credit reporting or other personal consumer re...,usc 1681i subject subsection except provide su...,0.00
3,Credit reporting or other personal consumer re...,file complaint equifax multiple serious violat...,-0.10
4,Credit reporting or other personal consumer re...,saferent solutions provide inaccurate informat...,0.00


In [10]:
labels = ['muito negativo', 'negativo', 'neutro', 'positivo']
valores = [-1, -0.5, 0, 0.5, 1]
pd.cut(df['escala'], bins=valores, labels=labels).value_counts()

escala
neutro            47000
negativo          45000
muito negativo    43246
positivo          10806
Name: count, dtype: int64

## Preparando um DF para utilizar o XGBoost Classifier

Como o dataset veio sem label da pipeline, apenas com a escala de -1 a 1, criamos aqui as labels.

In [11]:
def labelization(escala):
    if escala < -0.5:
        return 0
    elif (escala >= -0.5) & (escala < 0):
        return 1
    elif (escala >= 0) & (escala < 0.5):
        return 2
    else:
        return 3

In [12]:
df_class = df.copy()
df_class['label_real'] = df_class['escala'].apply(labelization)

In [13]:
df_class.head()

,product,processada_ml,escala,label_real
0,Credit reporting or other personal consumer re...,inquiries credit report recognize,0.00,2
1,Credit reporting or other personal consumer re...,write formally request block deletion account ...,-0.05,1
2,Credit reporting or other personal consumer re...,usc 1681i subject subsection except provide su...,0.00,2
3,Credit reporting or other personal consumer re...,file complaint equifax multiple serious violat...,-0.10,1
4,Credit reporting or other personal consumer re...,saferent solutions provide inaccurate informat...,0.00,2


In [14]:
df_class['label_real'].value_counts()

label_real
2    59761
0    43246
1    31519
3    11526
Name: count, dtype: int64

## Modelo de Classificação

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
import xgboost as xgb
from sklearn.metrics import mean_squared_error
import joblib
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score


# 1) Definição de features e target
X = df_class[['product', 'processada_ml']]  # Features (atributos)
y = df_class.label_real  # Target

# 2) Preparação: TF-IDF para texto + OneHot para Produto
text_vect = TfidfVectorizer(
    max_features=200_000,      
    ngram_range=(2,2),        
    min_df=2,
    max_df=0.9,
    strip_accents="unicode",
    lowercase=True
)

prod_enc = OneHotEncoder(handle_unknown="ignore")

preprocess = ColumnTransformer(
    transformers=[
        ("tf_idf", text_vect, "processada_ml"),
        ("prod", prod_enc, ["product"])
    ],
    remainder="drop", sparse_threshold=1.0
)

X = preprocess.fit_transform(X)
#salvamos o modelo de pré_processamento das features
joblib.dump(preprocess, 'preprocess_class.pkl')

# 3) Separação da base de treino e teste, embaralhando os dados e forçando uma estratificação conforme distribuição das labels
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True, stratify=y)


# 4) Classificador
reg = xgb.XGBClassifier()

# 5) Treino
reg.fit(X_train, y_train)#, reg__sample_weight=weights)
# variáveis utilizadas depois para explicação do modelo
xgb_importances = reg.feature_importances_ 
feature_names = preprocess.get_feature_names_out()

# 6) Predição e Avaliação
y_pred = reg.predict(X_test)

print(classification_report(y_test, y_pred))
print(accuracy_score(y_test, y_pred))
print("Matriz de confusão:\n", confusion_matrix(y_test, y_pred))

# 9) Salvar o modelo treinado
joblib.dump(reg, "modelo_treinado_class.pkl")

              precision    recall  f1-score   support

           0       0.81      0.67      0.73      8649
           1       0.79      0.52      0.63      6304
           2       0.65      0.89      0.75     11953
           3       0.91      0.64      0.75      2305

    accuracy                           0.72     29211
   macro avg       0.79      0.68      0.72     29211
weighted avg       0.75      0.72      0.72     29211

0.7239053781109855
Matriz de confusão:
 [[ 5769   428  2440    12]
 [  533  3279  2487     5]
 [  782   419 10626   126]
 [   52     9   772  1472]]


['modelo_treinado_class.pkl']